# Time & Space Complexity of ML Algorithms

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/algorithm-complexity)

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

Big-O describes how work *grows* with input size. This notebook makes that concrete two ways: first by **plotting** the growth classes, then by **timing** real algorithms and watching the measured curve match the theory.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
plt.style.use('dark_background')
rng = np.random.default_rng(0)

## 1. The growth classes

Each complexity class is just a function of the input size `n`. Plotted on a **log-scaled y-axis**, they separate cleanly — and $O(2^n)$ shoots off the top almost immediately.

In [ ]:
n = np.arange(1, 41)
classes = {
    'O(1)':        np.ones_like(n, dtype=float),
    'O(log n)':    np.log2(n),
    'O(n)':        n.astype(float),
    'O(n log n)':  n * np.log2(n),
    'O(n^2)':      n.astype(float)**2,
    'O(2^n)':      2.0**n,
}

plt.figure(figsize=(8, 5))
for name, y in classes.items():
    plt.plot(n, np.maximum(y, 1), label=name, linewidth=2)
plt.yscale('log')
plt.xlabel('input size n'); plt.ylabel('operations (log scale)')
plt.title('Big-O growth'); plt.legend(); plt.tight_layout(); plt.show()

## 2. Timing confirms the theory

Theory is only useful if it predicts reality. Let's **time** three operations across growing input sizes and check the shape of the measured curve:

- a single scan  → expect $O(n)$ (a straight line)
- sorting        → expect $O(n \log n)$ (almost straight)
- an all-pairs distance matrix → expect $O(n^2)$ (a parabola)

In [ ]:
def timeit(fn, *args, repeats=3):
    best = float('inf')
    for _ in range(repeats):
        t0 = time.perf_counter(); fn(*args); best = min(best, time.perf_counter() - t0)
    return best

sizes = [1000, 2000, 4000, 8000]
scan, sort, pairs = [], [], []
for m in sizes:
    x = rng.standard_normal(m)
    P = rng.standard_normal((m, 2))
    scan.append(timeit(lambda a: a.sum(), x))
    sort.append(timeit(np.sort, x))
    # all-pairs squared distances: O(m^2)
    pairs.append(timeit(lambda A: ((A[:, None] - A[None]) ** 2).sum(-1), P))

for name, ts in [('scan  O(n)', scan), ('sort  O(n log n)', sort), ('pairwise  O(n^2)', pairs)]:
    print(f'{name:20s}', ['%.4f' % t for t in ts])

Divide each timing by its predicted growth term — if the theory is right, the ratios should be roughly **flat** (constant work per predicted operation):

In [ ]:
s = np.array(sizes, dtype=float)
print('scan  / n      :', np.round(np.array(scan) / s, 8))
print('sort  / n log n :', np.round(np.array(sort) / (s * np.log2(s)), 9))
print('pairs / n^2    :', np.round(np.array(pairs) / s**2, 12))

## 3. KNN: lazy training, expensive prediction

KNN stores the data in $O(1)$ and pays $O(n\,d)$ *per query* at prediction time. Watch prediction time grow linearly with the training-set size:

In [ ]:
def knn_predict_all(X_train, y_train, X_query, k=5):
    # O(m * n * d): every query point vs every training point
    d = np.sqrt(((X_query[:, None, :] - X_train[None, :, :]) ** 2).sum(-1))
    idx = np.argsort(d, axis=1)[:, :k]
    return (y_train[idx].mean(axis=1) > 0.5).astype(int)

d, m_query = 10, 200
train_sizes = [1000, 2000, 4000, 8000]
pred_times = []
for ntr in train_sizes:
    Xtr = rng.standard_normal((ntr, d)); ytr = rng.integers(0, 2, ntr)
    Xq = rng.standard_normal((m_query, d))
    pred_times.append(timeit(knn_predict_all, Xtr, ytr, Xq))

plt.figure(figsize=(7, 4))
plt.plot(train_sizes, pred_times, 'o-', color='#6366f1', linewidth=2)
plt.xlabel('training set size n'); plt.ylabel('prediction time (s)')
plt.title('KNN prediction cost is linear in n'); plt.tight_layout(); plt.show()

## ✏️ Your turn

### Exercise 1 — classify a timing curve

The covariance-matrix step of PCA on `(n, d)` data costs $O(n\,d^2)$. Below, we fix `d` and grow `n`. **Fill in** the predicted growth term you'd divide by so the ratios come out flat, then check.

In [ ]:
d = 50
ns = [2000, 4000, 8000, 16000]
cov_times = []
for n_ in ns:
    X = rng.standard_normal((n_, d))
    cov_times.append(timeit(lambda A: (A.T @ A), X))

# TODO(you): with d fixed, forming X^T X is O(n * d^2), so as a function of n it is O(n).
# Divide by the predicted growth term in n so the ratios are ~flat.
predicted = None   # <-- replace with the np.array of predicted growth terms

# print(np.round(np.array(cov_times) / predicted, 10))

In [ ]:
# --- assert (passes silently when correct) ---
predicted = np.array(ns, dtype=float)   # O(n) with d held constant
ratios = np.array(cov_times) / predicted
# ratios should be roughly constant: max/min within a small factor
assert ratios.max() / ratios.min() < 4.0, 'ratios not flat — is the growth term right?'
print('OK — X^T X scales linearly in n when d is fixed')

<details><summary>Solution</summary>

With `d` held constant, $O(n\,d^2)$ is just $O(n)$ in the variable that changes. So divide by `n` (i.e. `np.array(ns)`); the ratios (seconds per row) are approximately constant. If you instead grew `d` with `n` fixed, you'd divide by `d**2`.

</details>

### Exercise 2 — the attention wall

Self-attention builds an $L \times L$ score matrix, so both its compute and memory are $O(L^2)$ in sequence length. Wall-clock timing of a BLAS matmul is too noisy at these sizes to see it cleanly, but the *number of score entries* is exact. **Fill in** the count of entries in the score matrix, then confirm doubling `L` quadruples it.

In [ ]:
def attention_scores(Q, K):
    # Q, K: (L, h) -> scores (L, L), an O(L^2 * h) operation
    return Q @ K.T

h = 64
Ls = [512, 1024, 2048]
entries = []
for L in Ls:
    Q = rng.standard_normal((L, h)); K = rng.standard_normal((L, h))
    S = attention_scores(Q, K)
    # TODO(you): how many scores does the (L, L) matrix hold? (hint: S.size)
    n_entries = None   # <-- replace
    entries.append(n_entries)

# print(entries)

In [ ]:
# --- assert (passes silently when correct) ---
entries = [L * L for L in Ls]   # the (L, L) score matrix holds L^2 entries
doubling_ratios = [entries[i+1] / entries[i] for i in range(len(entries)-1)]
print('entry-count ratios when L doubles:', doubling_ratios)
assert all(r == 4.0 for r in doubling_ratios), 'expected exactly 4x per doubling (quadratic)'
print('OK — attention memory scales quadratically in sequence length')

<details><summary>Solution</summary>

`n_entries = S.size`, which equals `L * L`. Doubling `L` multiplies the entry count by exactly $2^2 = 4$ — the quadratic wall, seen in memory rather than noisy wall-clock time. This is exactly why long-context models need FlashAttention (same $O(L^2)$ FLOPs, but $O(L)$ memory by never materializing the full matrix) or sparse/linear attention (which attack the time term itself).

</details>